<a href="https://colab.research.google.com/github/BhuvanHande/ALA/blob/main/regression_techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df_housing = pd.read_csv('/content/california_housing.csv')
display(df_housing.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/california_housing.csv'

## 1. Feature Engineering and Preprocessing

In [ ]:
# Make a copy to avoid modifying the original DataFrame
df = df_housing.copy()

# Create new features
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

# Display the new features and basic info
display(df[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].head())
print(df.info())

## 2. Handle Missing Values

In [ ]:
# Check for missing values
print("Missing values before handling:\n", df.isnull().sum())

# Fill missing 'total_bedrooms' with the median
df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)

# Recalculate 'bedrooms_per_room' after filling missing 'total_bedrooms'
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']

print("\nMissing values after handling:\n", df.isnull().sum())

Missing values before handling:
 longitude                     0
latitude                      0
housing_median_age            0
total_rooms                   0
total_bedrooms              207
population                    0
households                    0
median_income                 0
ocean_proximity               0
median_house_value            0
rooms_per_household           0
bedrooms_per_room           207
population_per_household      0
dtype: int64

Missing values after handling:
 longitude                   0
latitude                    0
housing_median_age          0
total_rooms                 0
total_bedrooms              0
population                  0
households                  0
median_income               0
ocean_proximity             0
median_house_value          0
rooms_per_household         0
bedrooms_per_room           0
population_per_household    0
dtype: int64


/tmp/ipykernel_1648/3283004166.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)


## 3. Outlier Detection (Anomaly Removal)

In [ ]:
import numpy as np

# Simple outlier removal based on domain knowledge or statistical methods
# For example, removing houses with unusually high bedrooms per room or population per household
# Using IQR for `bedrooms_per_room` as an example

Q1 = df['bedrooms_per_room'].quantile(0.25)
Q3 = df['bedrooms_per_room'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out outliers
df_filtered = df[(df['bedrooms_per_room'] >= lower_bound) & (df['bedrooms_per_room'] <= upper_bound)]

print(f"Original dataset size: {len(df)} rows")
print(f"Dataset size after removing outliers: {len(df_filtered)} rows")

df = df_filtered.copy()

Original dataset size: 20640 rows
Dataset size after removing outliers: 20005 rows


## 4. Data Splitting: Train, Validation, and Test Set (70:20:10)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separate features (X) and target (y)
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# Identify numerical and categorical features
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

# Create a preprocessing pipeline for numerical features (StandardScaler)
numerical_transformer = StandardScaler()

# Create a preprocessing pipeline for categorical features (OneHotEncoder)
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# First, split into training (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Then, split temp (30%) into validation (20% of total) and test (10% of total)
# 20% of total means (2/3) of temp; 10% of total means (1/3) of temp
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=1/3, random_state=42)

print(f"Training set size: {len(X_train)} samples")
print(f"Validation set size: {len(X_val)} samples")
print(f"Test set size: {len(X_test)} samples")

# Apply preprocessing separately to each set to avoid data leakage
# Note: fit_transform only on training data, transform on validation and test data
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

Training set size: 14003 samples
Validation set size: 4001 samples
Test set size: 2001 samples


## 5. Model Training: Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Initialize and train the Decision Tree Regressor
# We use a Regressor because 'median_house_value' is continuous, not categorical.
# If classification was strictly required, 'median_house_value' would need to be binned into categories.
dtree_regressor = DecisionTreeRegressor(random_state=42)
dtree_regressor.fit(X_train_processed, y_train)

print("Decision Tree Regressor trained successfully.")

Decision Tree Regressor trained successfully.


## 6. Model Evaluation: Regression Metrics and Cross-Validation

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

# Predict on the validation set
y_pred_val = dtree_regressor.predict(X_val_processed)

# Evaluate the model using regression metrics
mse = mean_squared_error(y_val, y_pred_val)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_val, y_pred_val)
r2 = r2_score(y_val, y_pred_val)

print("### Validation Set Metrics ###")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared (R2): {r2:.2f}")

print("\n--- Note on Classification Metrics ---")
print("Accuracy, F1-score, and Recall are metrics for classification problems.")
print("Since 'median_house_value' is a continuous target, we use regression metrics (MSE, RMSE, MAE, R2).")
print("If you intended to categorize house prices for classification, the target variable would need to be binned.")

# Perform Cross-Validation
# We will use the full training data (X_train_processed, y_train) for cross-validation
# The validation set was used for hyperparameter tuning decisions (not done yet, but typically).

print("\n### Cross-Validation ###")
cv_scores = cross_val_score(dtree_regressor, X_train_processed, y_train, cv=5, scoring='neg_mean_squared_error')
rmse_cv_scores = np.sqrt(-cv_scores)

print(f"Cross-validation RMSE scores: {rmse_cv_scores}")
print(f"Mean CV RMSE: {rmse_cv_scores.mean():.2f}")
print(f"Standard deviation of CV RMSE: {rmse_cv_scores.std():.2f}")

### Validation Set Metrics ###
Mean Squared Error (MSE): 4387112206.54
Root Mean Squared Error (RMSE): 66235.28
Mean Absolute Error (MAE): 43001.74
R-squared (R2): 0.67

--- Note on Classification Metrics ---
Accuracy, F1-score, and Recall are metrics for classification problems.
Since 'median_house_value' is a continuous target, we use regression metrics (MSE, RMSE, MAE, R2).
If you intended to categorize house prices for classification, the target variable would need to be binned.

### Cross-Validation ###
Cross-validation RMSE scores: [69842.43444579 67500.30157179 69110.3872575  69748.6153808
 69750.42212496]
Mean CV RMSE: 69190.43
Standard deviation of CV RMSE: 884.67


## 7. Addressing Overfitting and Underfitting

In [ ]:
print("### How to Identify and Address Overfitting/Underfitting ###")
print("\n**Overfitting** occurs when the model performs very well on the training data but poorly on unseen data (validation/test set). Signs:")
print("- Very high R2 on training set, but much lower R2 on validation/test set.")
print("- Training RMSE/MAE is significantly lower than validation/test RMSE/MAE.")

print("\n**Underfitting** occurs when the model performs poorly on both training and unseen data. Signs:")
print("- Low R2 on both training and validation/test sets.")
print("- High RMSE/MAE on both training and validation/test sets.")

print("\n**Techniques to overcome Overfitting/Underfitting for Decision Tree Regression:**\n")

print("**1. Hyperparameter Tuning (Most Common for Decision Trees):**")
print("   - `max_depth`: Limit the maximum depth of the tree. A shallower tree reduces overfitting.")
print("   - `min_samples_split`: The minimum number of samples required to split an internal node. Increasing this value prevents the tree from learning highly specific patterns.")
print("   - `min_samples_leaf`: The minimum number of samples required to be at a leaf node. Similar to `min_samples_split`.")
print("   - `max_features`: The number of features to consider when looking for the best split. Restricting this can introduce more randomness and reduce overfitting.")
print("   - Use techniques like `GridSearchCV` or `RandomizedSearchCV` with cross-validation to find the optimal combination of these hyperparameters.")

print("**2. Ensemble Methods:**")
print("   - **Bagging (e.g., RandomForestRegressor):** Trains multiple decision trees on different subsets of the data and averages their predictions. Highly effective at reducing variance (overfitting).")
print("   - **Boosting (e.g., GradientBoostingRegressor, XGBoost, LightGBM):** Builds trees sequentially, with each new tree correcting the errors of the previous one. Can reduce bias and variance.")

print("**3. Feature Engineering:**")
print("   - Create more informative features or remove irrelevant/noisy ones. This can help both underfitting (by providing better signals) and overfitting (by simplifying the model's task).")

print("**4. More Data (for Underfitting):**")
print("   - If the model is underfitting due to lack of diverse training examples, acquiring more data can help the model learn more general patterns.")

print("**Next Steps:** You can use techniques like `GridSearchCV` or `RandomizedSearchCV` to systematically tune the hyperparameters of the `DecisionTreeRegressor` or explore more advanced ensemble models like `RandomForestRegressor` to improve performance and manage overfitting/underfitting.")

### How to Identify and Address Overfitting/Underfitting ###

**Overfitting** occurs when the model performs very well on the training data but poorly on unseen data (validation/test set). Signs:
- Very high R2 on training set, but much lower R2 on validation/test set.
- Training RMSE/MAE is significantly lower than validation/test RMSE/MAE.

**Underfitting** occurs when the model performs poorly on both training and unseen data. Signs:
- Low R2 on both training and validation/test sets.
- High RMSE/MAE on both training and validation/test sets.

**Techniques to overcome Overfitting/Underfitting for Decision Tree Regression:**

**1. Hyperparameter Tuning (Most Common for Decision Trees):**
   - `max_depth`: Limit the maximum depth of the tree. A shallower tree reduces overfitting.
   - `min_samples_split`: The minimum number of samples required to split an internal node. Increasing this value prevents the tree from learning highly specific patterns.
   - `min_samples_leaf`: The minimum

## Addressing Overfitting and Underfitting with Hyperparameter Tuning (Code Example)

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
# 'max_depth' controls overfitting (deeper trees can overfit)
# 'min_samples_leaf' controls overfitting (smaller values can overfit)
param_grid = {
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 4, 8, 16]
}

# Initialize the Decision Tree Regressor
dtree_base = DecisionTreeRegressor(random_state=42)

# Initialize GridSearchCV
# We'll use the training data for cross-validation within GridSearchCV
# scoring='neg_mean_squared_error' is used because GridSearchCV maximizes scores,
# and MSE is a loss function (lower is better), so we negate it.
grid_search = GridSearchCV(
    dtree_base,
    param_grid,
    cv=5, # 5-fold cross-validation
    scoring='neg_mean_squared_error',
    n_jobs=-1, # Use all available CPU cores
    verbose=1
)

# Fit GridSearchCV on the preprocessed training data
grid_search.fit(X_train_processed, y_train)

# Get the best parameters and best score
best_params = grid_search.best_params_
best_rmse_cv = np.sqrt(-grid_search.best_score_)

print(f"Best Hyperparameters: {best_params}")
print(f"Best Cross-validated RMSE: {best_rmse_cv:.2f}")

# Get the best model
best_dtree_model = grid_search.best_estimator_

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best Hyperparameters: {'max_depth': 15, 'min_samples_leaf': 16}
Best Cross-validated RMSE: 58539.46


### Evaluate the Tuned Model on the Test Set

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Predict on the unseen test set using the best model
y_pred_test = best_dtree_model.predict(X_test_processed)

# Evaluate the tuned model's performance on the test set
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print("### Tuned Model Test Set Metrics ###")
print(f"Mean Squared Error (MSE): {mse_test:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_test:.2f}")
print(f"Mean Absolute Error (MAE): {mae_test:.2f}")
print(f"R-squared (R2): {r2_test:.2f}")

print("\nCompare these test set metrics to the validation metrics from the initial model and the cross-validated RMSE.")
print("Improvements in these metrics, especially a lower RMSE and higher R2 on the test set,")
print("indicate that hyperparameter tuning has likely helped in reducing overfitting and improving generalization.")

NameError: name 'best_dtree_model' is not defined

## Demonstrating and Addressing Underfitting

Underfitting typically occurs when a model is too simple to capture the underlying patterns in the data. For Decision Trees, this often means the tree is not deep enough or is overly constrained.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

print("### Demonstrating Underfitting (Very Simple Decision Tree) ###")

# Train a Decision Tree with intentionally low max_depth to cause underfitting
dtree_underfit = DecisionTreeRegressor(max_depth=3, random_state=42)
dtree_underfit.fit(X_train_processed, y_train)

# Predict on training and test sets
y_train_pred_underfit = dtree_underfit.predict(X_train_processed)
y_test_pred_underfit = dtree_underfit.predict(X_test_processed)

# Evaluate training set performance
mse_train_underfit = mean_squared_error(y_train, y_train_pred_underfit)
rmse_train_underfit = np.sqrt(mse_train_underfit)
r2_train_underfit = r2_score(y_train, y_train_pred_underfit)

# Evaluate test set performance
mse_test_underfit = mean_squared_error(y_test, y_test_pred_underfit)
rmse_test_underfit = np.sqrt(mse_test_underfit)
r2_test_underfit = r2_score(y_test, y_test_pred_underfit)

print(f"\nTraining Set RMSE (Underfit): {rmse_train_underfit:.2f}")
print(f"Training Set R2 (Underfit): {r2_train_underfit:.2f}")
print(f"Test Set RMSE (Underfit): {rmse_test_underfit:.2f}")
print(f"Test Set R2 (Underfit): {r2_test_underfit:.2f}")

print("\nAs you can see, both training and test set R2 scores are significantly lower than our tuned model (R2 ~0.76).")
print("This indicates the model is too simple and is underfitting the data.")

### Addressing Underfitting

To overcome underfitting in a Decision Tree, we need to allow the model to be more complex and learn more nuanced patterns from the data. The primary ways to do this are by:

1.  **Increasing `max_depth`**: Allowing the tree to grow deeper. A deeper tree can capture more intricate relationships.
2.  **Decreasing `min_samples_leaf` / `min_samples_split`**: Allowing the tree to make more splits, even with fewer samples in a node. This makes the tree more granular.

Our previous **Hyperparameter Tuning using `GridSearchCV`** (cell `93ac8f77`) already addressed both underfitting and overfitting by systematically searching for the best combination of `max_depth` and `min_samples_leaf`. The optimized parameters (`max_depth: 15`, `min_samples_leaf: 16`) found by `GridSearchCV` allowed the model to achieve a better balance between bias (underfitting) and variance (overfitting), leading to the improved performance observed on the test set (R2: 0.76).

Essentially, the code you ran for hyperparameter tuning is the method to address both underfitting (by making the model sufficiently complex) and overfitting (by preventing it from becoming *too* complex).